```mermaid
graph LR
 日本語+Pythonのコーパス作成 --> トークナイザーの学習
 トークナイザーの学習 --> GPTモデルの設計
  GPTモデルの設計 --> 学習ループの実装
  学習ループの実装 --> ファインチューニング
  ファインチューニング --> 推論
```

| Step | Name |Description |
| ---- | ----------- |----------- |
| 1 | 日本語+Pythonのコーパス作成 | (1)日本語(20~200MB) --> Wikipedia, 青空文庫, ニュース系コーパス, (2)Python(20~200MB) --> GitHub, Kaggle Notebooks |
|2 | トークナイザーの学習 | 語彙数(16k~32k), 日本語(BPE < Unigram), Python code(インデントや記号を分離しすぎない) |
|3| GPTモデルの設計 | Embedding(token embedding, position embedding), Transformer Block(Multi-Head Attention, MLP, Layer Normalization, Residual), Language Model Heads(Softmax) |
|4| 学習ループの実装 | batch size(16~48), sequence length(256~512), learning time(1h~1day) |
|5| ファインチューニング | Kaggle Notebooks, Python code, Python Q&A |
|6| 推論 ||

In [64]:
from datasets import load_dataset
import pandas as pd
import os

In [65]:
# Japanese datasets 
ds_wiki = load_dataset("mini97/filtered_japanese-wikipedia") 
ds_aozora = load_dataset("globis-university/aozorabunko-clean") 

# Python datasets 
ds_python = load_dataset("Arjun-G-Ravi/Python-codes")

In [66]:
print(type(ds_wiki))
print(type(ds_aozora))
print(type(ds_python))

<class 'datasets.dataset_dict.DatasetDict'>
<class 'datasets.dataset_dict.DatasetDict'>
<class 'datasets.dataset_dict.DatasetDict'>


In [67]:
ds_wiki.column_names

{'train': ['text',
  'meta',
  'original',
  'mean_paragraph_length',
  'num_paragraphs',
  'short_paragraph_count',
  'total_length',
  'main_language_ratio']}

In [68]:
ds_wiki["train"][0]

{'text': '『勝つか死ぬか』はHBO(日本ではスター・チャンネルが放送)のファンタジー・ドラマ・シリーズである『ゲーム・オブ・スローンズ』の第1章『七王国戦記』の第7話である。プロデューサーでもあるデイヴィッド・ベニオフ と D・B・ワイスが脚本を書き、 ダニエル・ミナハンが監督した。\n\n本エピソードでは、七王国の政治バランスの崩壊がさらに進み、ロバート王が狩りで外出している間に、エダードが発見した事実をサーセイに明らかにする。タイトルはサーセイの言葉「王座争奪戦では勝つか死ぬかです。妥協点はありません。」の引用である。このキャッチフレーズは原作本およびTVシリーズのプロモーションで多用されたものである。\n\nあらすじ\n\nラニスターの陣\nタイウィン・ラニスター公(チャールズ・ダンス)は牡鹿の皮をはぎながら、息子のジェイミー(ニコライ・コスター＝ワルドー)と話す。スターク家との争いを起こしたことを責めながらも、ラニスター家が七王国を統治する王朝を築く絶好の機会であるとタイウィンは信じる。キャトリンがティリオンを逮捕したことへの復讐として、軍の半分をジェイミーに与えて、タリー家の本拠でレディ・キャトリンの生家であるリヴァーランを攻めさせる。\n\nウィンターフェル\n捕えられた〈野人〉のオシャ(ナタリア・テナ)はスターク家の召使となり、シオン(アルフィー・アレン)に、もしもシオンの故郷の鉄諸島で捕えられていたならもっとひどい目に会っていたはずだとからかわれる。オシャに迫ろうとしたシオンを目撃したメイスター・ルイーウィンはシオンを追い払い、なぜ〈野人〉たちが〈壁〉の南に逃げてくるのかとオシャに問う。オシャは数千年の眠りから覚めたホワイト・ウォーカーから逃げてきたのだと言い、七王国のすべての軍は北に進軍してこの脅威に対処すべきだと言う。',
 'meta': {'id': '2969837',
  'title': '勝つか死ぬか',
  'url': 'https://ja.wikipedia.org/wiki/%E5%8B%9D%E3%81%A4%E3%81%8B%E6%AD%BB%E3%81%AC%E3%81%8B'},
 'original': '『勝つか死ぬか』はHBO(日本ではスター・チャンネルが放送)のファンタジー・ドラマ・シリーズである『

In [69]:
ds_aozora.column_names

{'train': ['text', 'footnote', 'meta']}

In [70]:
ds_aozora["train"][0]

{'text': '深いおどろきにうたれて、\n名高いウェストミンスターに\n真鍮や石の記念碑となって\nすべての王侯貴族が集まっているのをみれば、\n今はさげすみも、ほこりも、見栄もない。\n善にかえった貴人の姿、\n華美と俗世の権勢をすてた\nけがれのない帝王の姿がみえるではないか。\nいろどられた、おもちゃのような墓石に\n今は静かに物云わぬ魂がどんなに満足していることか。\nかつてはその足にふまえた全世界をもってしても\nその欲望を満たすこともおさえることも出来なかったのに。\n生とは冷たい幸福の結ぶ氷であり、\n死とはあらゆる人間の虚栄をとかす霜解けである。\n――「クリストレロの諷刺詩」一五九八年、Ｔ・Ｂ作\n\n\n\n\u3000秋も更けて、暁闇がすぐに黄昏となり、暮れてゆく年に憂愁をなげかけるころの、おだやかな、むしろ物さびしいある日、わたしはウェストミンスター寺院を逍遥して数時間すごしたことがある。悲しげな古い大伽藍の荘厳さには、この季節の感覚になにかぴったりするものがあった。その入口を通ったとき、わたしは、昔の人の住む国に逆もどりし、過ぎ去った時代の闇のなかに身を没してゆくような気がした。\n\u3000わたしはウェストミンスター・スクールの中庭から入り、低い円天井の長い廊下を通って行ったが、そこは巨大な壁にあけられた円形の穴でかすかに一部分が明るくなっているだけなので、あたかも地下に潜ったような感じがした。この暗い廊下を通して廻廊が遠くに見え、聖堂守の老人の黒い衣をまとった姿が、うす暗い円天井の下に動き、近くの墓地からぬけ出してきた幽霊のように見えた。\n\u3000こういう陰鬱な僧院の跡を通って寺院に近づいてゆくと、おのずから厳粛な思索にふさわしい気持ちになるものである。廻廊は昔ながらの世間を遠ざかった静寂の面影をいまだにとどめている。灰色の壁は湿気のために色があせ、歳月を経て崩れおちそうになっている。白い苔の衣が壁にはめこんだ記念碑の碑文をおおい、髑髏や、そのほかの葬儀の表象をもかくしている。鋭く刻んだ鑿のあとは、精巧な彫刻をほどこしたアーチの狭間飾りからすでに消え去っている。薔薇の模様がかなめ石を飾っていたが、その美しく茂った姿はなくなってしまっている。あらゆるものが、幾星霜のおもむろな侵蝕のあとをとどめている。だが、そのほろびのなかに

In [71]:
ds_python.column_names

{'train': ['code', 'question']}

In [72]:
ds_python["train"][0]

{'code': 'arr = [2, 4, 6, 8, 10]',
 'question': 'Create an array of length 5 which contains all even numbers between 1 and 10.'}

In [73]:
ds_wiki.num_rows

{'train': 3051822}

In [74]:
ds_aozora.num_rows

{'train': 16951}

In [75]:
ds_python.num_rows

{'train': 13815}

In [76]:
dataset = [ds_wiki, ds_aozora, ds_python]
len(dataset)

3

In [77]:
ds_wiki["train"][0]["text"]

'『勝つか死ぬか』はHBO(日本ではスター・チャンネルが放送)のファンタジー・ドラマ・シリーズである『ゲーム・オブ・スローンズ』の第1章『七王国戦記』の第7話である。プロデューサーでもあるデイヴィッド・ベニオフ と D・B・ワイスが脚本を書き、 ダニエル・ミナハンが監督した。\n\n本エピソードでは、七王国の政治バランスの崩壊がさらに進み、ロバート王が狩りで外出している間に、エダードが発見した事実をサーセイに明らかにする。タイトルはサーセイの言葉「王座争奪戦では勝つか死ぬかです。妥協点はありません。」の引用である。このキャッチフレーズは原作本およびTVシリーズのプロモーションで多用されたものである。\n\nあらすじ\n\nラニスターの陣\nタイウィン・ラニスター公(チャールズ・ダンス)は牡鹿の皮をはぎながら、息子のジェイミー(ニコライ・コスター＝ワルドー)と話す。スターク家との争いを起こしたことを責めながらも、ラニスター家が七王国を統治する王朝を築く絶好の機会であるとタイウィンは信じる。キャトリンがティリオンを逮捕したことへの復讐として、軍の半分をジェイミーに与えて、タリー家の本拠でレディ・キャトリンの生家であるリヴァーランを攻めさせる。\n\nウィンターフェル\n捕えられた〈野人〉のオシャ(ナタリア・テナ)はスターク家の召使となり、シオン(アルフィー・アレン)に、もしもシオンの故郷の鉄諸島で捕えられていたならもっとひどい目に会っていたはずだとからかわれる。オシャに迫ろうとしたシオンを目撃したメイスター・ルイーウィンはシオンを追い払い、なぜ〈野人〉たちが〈壁〉の南に逃げてくるのかとオシャに問う。オシャは数千年の眠りから覚めたホワイト・ウォーカーから逃げてきたのだと言い、七王国のすべての軍は北に進軍してこの脅威に対処すべきだと言う。'

In [80]:
ds_wiki["train"].num_rows

3051822

In [78]:
# delete and create new corpus.txt
if os.path.exists("corpus.txt"):
    os.remove("corpus.txt")
    os.makedirs("corpus.txt")

In [ ]:
# wiki
with open("corpus.txt", "w", encoding="utf-8") as f:
    for row in ds_wiki["train"]:
        text = row["original"]
        text = text.replace("\n", " ")
        f.write(text + "\n")
    
# aozora
with open("corpus.txt", "w", encoding="utf-8") as f:
    for row in range(len(ds_aozora["train"].num_rows)):
        text = ds_aozora["train"][row]["text"]
        text = text.replace("\n", " ")
        f.write(text + "\n")

# python
with open("corpus.txt", "w", encoding="utf-8") as f:
    for row in range(len(ds_python["train"].num_rows)):
        code = ds_python["train"][row]["code"]
        text = ds_python["train"][row]["question"]
        f.write("<code>" + "\n" + code + "\n" + text + "\n" + "</code>" + "\n")

SyntaxError: invalid syntax. Perhaps you forgot a comma? (3798753776.py, line 10)